# Supplementary Figure S3 — score architecture stress tests

This notebook compares the frozen Equal4 score with PC1, Core3, SLC7A11,
and the four individual genes without changing weights or selecting models
from outcomes.


In [1]:
from pathlib import Path
import hashlib
import json
import os
import re

import numpy as np
import pandas as pd
import pyreadr
import statsmodels.api as sm
from scipy.stats import spearmanr
from sklearn.decomposition import PCA

PROJECT_ROOT = Path(
    os.environ.get("MITOXY_PROJECT_ROOT", "/scratch/ar58064/mitoxy")
).resolve()
MANIFEST_PATH = (
    PROJECT_ROOT / "data" / "provenance" /
    "paper_public_source_manifest.csv"
)
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "figure_3_clean"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GENES = ["BAK1", "BAX", "BID", "SLC7A11"]
CORE3 = ["BAK1", "BAX", "BID"]
manifest = pd.read_csv(MANIFEST_PATH)


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def source_path(dataset, role):
    match = manifest.loc[
        manifest["dataset"].eq(dataset) & manifest["role"].eq(role)
    ]
    if len(match) != 1:
        raise RuntimeError(
            f"Expected one source for ({dataset!r}, {role!r}); found {len(match)}"
        )
    row = match.iloc[0]
    path = PROJECT_ROOT / str(row["relative_path"])
    if not path.is_file():
        raise FileNotFoundError(path)
    if path.stat().st_size != int(row["size_bytes"]):
        raise RuntimeError(f"Source-size mismatch: {path}")
    if sha256_file(path) != str(row["sha256"]):
        raise RuntimeError(f"Source SHA-256 mismatch: {path}")
    return path


RAW = {
    "tracerx_tpm": source_path("TRACERx Renal", "regional tumor TPM"),
    "tracerx_annotation": source_path("TRACERx Renal", "regional annotation"),
    "tracerx_clinical": source_path("TRACERx Renal", "clinical annotation"),
    "tracerx_supplement": source_path(
        "TRACERx Renal", "evolutionary/genomic supplement"
    ),
    "cptac_rna": source_path("CPTAC ccRCC", "tumor RNA expression"),
    "cptac_protein": source_path(
        "CPTAC ccRCC", "tumor protein abundance"
    ),
    "cptac_clinical": source_path("CPTAC ccRCC", "clinical supplement"),
    "cptac_phospho": source_path(
        "CPTAC ccRCC", "phosphoproteomics package export"
    ),
}

display(pd.DataFrame({"analysis_key": RAW.keys(), "path": RAW.values()}))
print("Public-source integrity audit: PASS")


,analysis_key,path
0,tracerx_tpm,/scratch/ar58064/mitoxy/data/raw/tracerx_renal...
1,tracerx_annotation,/scratch/ar58064/mitoxy/data/raw/tracerx_renal...
2,tracerx_clinical,/scratch/ar58064/mitoxy/data/raw/tracerx_renal...
3,tracerx_supplement,/scratch/ar58064/mitoxy/data/raw/tracerx_renal...
4,cptac_rna,/scratch/ar58064/mitoxy/data/raw/cptac_ccrcc_p...
5,cptac_protein,/scratch/ar58064/mitoxy/data/raw/cptac_ccrcc_p...
6,cptac_clinical,/scratch/ar58064/mitoxy/data/raw/cptac_ccrcc_p...
7,cptac_phospho,/scratch/ar58064/mitoxy/data/raw/cptac_ccrcc_p...


Public-source integrity audit: PASS


In [2]:
def zscore(values):
    values = pd.Series(values, index=getattr(values, "index", None), dtype=float)
    return (values - values.mean()) / values.std(ddof=0)


def parse_grade(value):
    if pd.isna(value):
        return np.nan
    match = re.search(r"(?:G(?:RADE)?\s*)?([1-4])", str(value), flags=re.I)
    return float(match.group(1)) if match else np.nan


def parse_stage(value):
    if pd.isna(value):
        return np.nan
    text = str(value).upper().replace("STAGE", " ")
    match = re.search(r"\b(IV|III|II|I)\b", text)
    if match:
        return {"I": 1., "II": 2., "III": 3., "IV": 4.}[match.group(1)]
    match = re.search(r"\b([1-4])\b", text)
    return float(match.group(1)) if match else np.nan


def normalized(text):
    return re.sub(r"[^a-z0-9]+", " ", str(text).lower()).strip()


def find_column(columns, include, exclude=(), required=True):
    include = [normalized(x) for x in include]
    exclude = [normalized(x) for x in exclude]
    candidates = []
    for column in columns:
        name = normalized(column)
        if all(term in name for term in include) and not any(
            term in name for term in exclude
        ):
            candidates.append(column)
    if not candidates:
        if required:
            raise KeyError(
                f"No column matched include={include}, exclude={exclude}. "
                f"Available columns: {list(columns)}"
            )
        return None
    return sorted(candidates, key=lambda x: (len(str(x)), str(x)))[0]


def read_linkedomics(path):
    matrix = pd.read_csv(path, sep="\t", index_col=0, low_memory=False)
    matrix = matrix.apply(pd.to_numeric, errors="coerce")
    matrix.index = [str(x).split("|")[0].strip().upper() for x in matrix.index]
    matrix.columns = [str(x).strip().upper() for x in matrix.columns]
    if matrix.index.duplicated().any():
        matrix = matrix.groupby(level=0, sort=False).mean()
    return matrix


def fit_hc3(data, outcome, predictors, focal):
    fields = [outcome, *predictors]
    complete = data[fields].replace([np.inf, -np.inf], np.nan).dropna()
    design = sm.add_constant(complete[predictors], has_constant="add")
    model = sm.OLS(complete[outcome].astype(float), design.astype(float)).fit(
        cov_type="HC3", use_t=False
    )
    ci = model.conf_int().loc[focal]
    return {
        "n": len(complete), "beta": float(model.params[focal]),
        "se": float(model.bse[focal]), "p_value": float(model.pvalues[focal]),
        "ci_low": float(ci.iloc[0]), "ci_high": float(ci.iloc[1]),
    }


def fit_logit(data, outcome, predictors, focal):
    complete = data[[outcome, *predictors]].replace(
        [np.inf, -np.inf], np.nan
    ).dropna()
    design = sm.add_constant(complete[predictors], has_constant="add")
    model = sm.Logit(
        complete[outcome].astype(float), design.astype(float)
    ).fit(disp=False)
    ci = model.conf_int().loc[focal]
    return {
        "n": len(complete), "beta": float(model.params[focal]),
        "se": float(model.bse[focal]), "p_value": float(model.pvalues[focal]),
        "ci_low": float(ci.iloc[0]), "ci_high": float(ci.iloc[1]),
    }


In [3]:
expression = pyreadr.read_r(str(RAW["tracerx_tpm"]))[None]
expression.index = expression.index.astype(str).str.upper()
expression.columns = expression.columns.astype(str)
missing = sorted(set(GENES) - set(expression.index))
assert not missing, f"TRACERx expression is missing: {missing}"

annotation = pd.read_csv(RAW["tracerx_annotation"], sep="\t", dtype=str)
sample_column = find_column(annotation.columns, ["sample"])
patient_column = find_column(annotation.columns, ["patient"])
type_column = find_column(annotation.columns, ["type", "collapsed"])

annotation[sample_column] = annotation[sample_column].astype(str).str.strip()
annotation[patient_column] = annotation[patient_column].astype(str).str.strip()
primary_annotation = annotation.loc[
    annotation[type_column].astype(str).str.upper().eq("PRIMARY")
].copy()
primary_samples = [
    sample for sample in primary_annotation[sample_column]
    if sample in expression.columns
]
assert len(primary_samples) > 0

# Figure 3's architecture and component analyses use the public TPM values
# directly before gene-wise standardization. This differs from the log-TPM
# transformation used for the cross-cohort rank analysis in Figure 1.
region_values = (
    expression.loc[GENES, primary_samples]
    .T.apply(pd.to_numeric, errors="coerce")
)
if region_values.isna().any().any():
    raise ValueError("Missing target-gene TPM values in primary regions")
region_z = region_values.apply(zscore, axis=0)
region_z.columns = [f"z_{gene}" for gene in GENES]
region_z["sample_id"] = region_z.index
region_z = region_z.merge(
    primary_annotation[[sample_column, patient_column]],
    left_on="sample_id", right_on=sample_column,
    how="left", validate="one_to_one",
)
region_z = region_z.rename(columns={patient_column: "patient"})

patient = region_z.groupby("patient")[[f"z_{g}" for g in GENES]].mean()
patient.index = patient.index.astype(str)
patient.index.name = "patient"

# Patient-constant annotation fields.
condition_column = find_column(annotation.columns, ["condition"])
patient_annotation = (
    primary_annotation[[patient_column, condition_column]]
    .groupby(patient_column, as_index=False).first()
    .rename(columns={patient_column: "patient", condition_column: "condition"})
)

clinical = pd.read_csv(RAW["tracerx_clinical"], sep="\t", header=1, dtype=str)
clinical.columns = [str(x).strip() for x in clinical.columns]
clinical_patient = find_column(clinical.columns, ["subject"])
clinical_grade = find_column(clinical.columns, ["fuhrman", "grade"])
clinical_stage = find_column(clinical.columns, ["overall", "stage"])
clinical_age = find_column(clinical.columns, ["age"])
clinical_leibovich = find_column(clinical.columns, ["leibovich"])
clinical_table = pd.DataFrame({
    "patient": clinical[clinical_patient].astype(str).str.strip(),
    "grade": clinical[clinical_grade].map(parse_grade),
    "stage": clinical[clinical_stage].map(parse_stage),
    "age": pd.to_numeric(clinical[clinical_age], errors="coerce"),
    "leibovich": pd.to_numeric(clinical[clinical_leibovich], errors="coerce"),
}).drop_duplicates("patient")

patient = (
    patient.reset_index()
    .merge(clinical_table, on="patient", how="left", validate="one_to_one")
    .merge(patient_annotation, on="patient", how="left", validate="one_to_one")
)
patient["punctuated"] = patient["condition"].astype(str).str.casefold().eq(
    "punctuated"
).astype(float)

# These covariates are carried in the public regional annotation. Their exact
# labels differ slightly across bundle releases, so resolve them by meaning.
purity_column = find_column(annotation.columns, ["purity"])
wgi_column = find_column(annotation.columns, ["wgi"], required=False)
if wgi_column is None:
    wgi_column = find_column(annotation.columns, ["weighted", "genome"], required=False)
ith_column = find_column(annotation.columns, ["ith"], required=False)
if ith_column is None:
    ith_column = find_column(annotation.columns, ["intratum", "heterogeneity"])

covariate_rows = primary_annotation[[patient_column, purity_column, wgi_column, ith_column]].copy()
covariate_rows = covariate_rows.rename(columns={
    patient_column: "patient", purity_column: "purity",
    wgi_column: "wgi", ith_column: "ith",
})
for column in ["purity", "wgi", "ith"]:
    covariate_rows[column] = pd.to_numeric(covariate_rows[column], errors="coerce")
covariates = covariate_rows.groupby("patient", as_index=False).mean(numeric_only=True)
patient = patient.merge(covariates, on="patient", how="left", validate="one_to_one")

required_complete = ["grade", "stage", "age", "purity", "condition", "wgi", "ith"]
patient = patient.dropna(subset=required_complete).copy()
assert patient["patient"].is_unique
assert set(patient["punctuated"].unique()).issubset({0., 1.})
print("TRACERx complete-case patients:", len(patient))
display(patient[required_complete].describe(include="all"))


TRACERx complete-case patients: 73


,grade,stage,age,purity,condition,wgi,ith
count,73.000000,73.000000,73.000000,73.000000,73,73.000000,73.000000
unique,NaN,NaN,NaN,NaN,4,NaN,NaN
top,NaN,NaN,NaN,NaN,Other,NaN,NaN
freq,NaN,NaN,NaN,NaN,31,NaN,NaN
mean,3.027397,2.821918,63.328767,0.448575,NaN,0.297906,1.690188
std,0.744845,1.182540,9.248205,0.134582,NaN,0.201328,2.201832
min,1.000000,1.000000,34.000000,0.195000,NaN,0.040520,0.000000
25%,3.000000,2.000000,58.000000,0.350000,NaN,0.131545,0.333333
50%,3.000000,3.000000,64.000000,0.428000,NaN,0.235950,1.142857
75%,4.000000,4.000000,69.000000,0.550000,NaN,0.454675,2.000000


In [4]:
gene_columns = [f"z_{gene}" for gene in GENES]
# Standardize the patient-level regional means so every score architecture and
# single-gene model is expressed per patient-level SD.
gene_matrix = patient[gene_columns].apply(zscore, axis=0)
assert gene_matrix.notna().all().all()

pca = PCA(n_components=1, svd_solver="full")
pc1_values = pca.fit_transform(gene_matrix).ravel()
loadings = pca.components_[0].copy()
if loadings.mean() < 0:
    loadings *= -1
    pc1_values *= -1
panel_a = pd.DataFrame({"gene": GENES, "PC1_loading": loadings})

model_data = patient.copy()
for column in gene_columns:
    model_data[column] = gene_matrix[column]

model_data["Equal4"] = gene_matrix.mean(axis=1)
model_data["PCA1_4"] = pc1_values
model_data["BAK1_centered"] = (
    0.5 * gene_matrix["z_BAK1"]
    + 0.5 * gene_matrix[["z_BAX", "z_BID", "z_SLC7A11"]].mean(axis=1)
)
model_data["Apoptotic3"] = gene_matrix[
    ["z_BAK1", "z_BAX", "z_BID"]
].mean(axis=1)
model_data["BAK1_SLC7A11"] = gene_matrix[
    ["z_BAK1", "z_SLC7A11"]
].mean(axis=1)
model_data["BAK1_only"] = gene_matrix["z_BAK1"]
model_data["Rest3"] = gene_matrix[
    ["z_BAX", "z_BID", "z_SLC7A11"]
].mean(axis=1)

score_names = [
    "Equal4", "PCA1_4", "BAK1_centered", "Apoptotic3",
    "BAK1_SLC7A11", "BAK1_only",
]
architecture_rows = []
for score in score_names:
    model_data[f"{score}_sd"] = zscore(model_data[score])
    grade = fit_hc3(
        model_data, "grade", [f"{score}_sd", "purity", "age"],
        f"{score}_sd"
    )
    stage = fit_hc3(
        model_data, "stage", [f"{score}_sd", "purity", "age"],
        f"{score}_sd"
    )
    punct = fit_hc3(
        model_data, f"{score}_sd",
        ["punctuated", "purity", "age", "grade", "stage", "wgi", "ith"],
        "punctuated",
    )
    architecture_rows.append({
        "score": score, "grade_beta": grade["beta"],
        "stage_beta": stage["beta"], "punctuated_beta": punct["beta"],
        "grade_p": grade["p_value"], "stage_p": stage["p_value"],
        "punctuated_p": punct["p_value"],
    })
panel_b = pd.DataFrame(architecture_rows)
panel_b["score_label"] = panel_b["score"].map({
    "Equal4": "Equal4", "PCA1_4": "PCA1",
    "BAK1_centered": "BAK1-centered", "Apoptotic3": "Core3",
    "BAK1_SLC7A11": "BAK1 + SLC7A11", "BAK1_only": "BAK1 only",
})

single_rows = []
for component, column in [
    ("BAK1", "z_BAK1"), ("BAX", "z_BAX"), ("BID", "z_BID"),
    ("SLC7A11", "z_SLC7A11"), ("Equal4", "Equal4_sd"),
]:
    result = fit_hc3(
        model_data, "grade", [column, "purity", "age"], column
    )
    single_rows.append({"component": component, **result})
panel_d = pd.DataFrame(single_rows)

joint_grade = fit_hc3(
    model_data, "grade", ["z_BAK1", "Rest3", "purity", "age"], "z_BAK1"
)
joint_grade_rest = fit_hc3(
    model_data, "grade", ["z_BAK1", "Rest3", "purity", "age"], "Rest3"
)
logit_predictors = [
    "z_BAK1", "Rest3", "purity", "age", "grade", "stage", "wgi", "ith"
]
joint_punct = fit_logit(
    model_data, "punctuated", logit_predictors, "z_BAK1"
)
joint_punct_rest = fit_logit(
    model_data, "punctuated", logit_predictors, "Rest3"
)
panel_e = pd.DataFrame([
    {"endpoint": "Grade", "component": "BAK1", **joint_grade},
    {"endpoint": "Grade", "component": "Rest3", **joint_grade_rest},
    {"endpoint": "Punctuated evolution", "component": "BAK1", **joint_punct},
    {"endpoint": "Punctuated evolution", "component": "Rest3", **joint_punct_rest},
])

for table in [panel_a, panel_b, panel_d, panel_e]:
    numeric = table.select_dtypes(include=[np.number])
    assert np.isfinite(numeric).all().all()
for table in [panel_d, panel_e]:
    assert (table["ci_low"] <= table["beta"]).all()
    assert (table["beta"] <= table["ci_high"]).all()
    assert table["p_value"].between(0, 1).all()

panel_a.to_csv(OUTPUT_DIR / "panel_a_pc1_loadings.csv", index=False)
panel_b.to_csv(OUTPUT_DIR / "panel_b_score_architectures.csv", index=False)
panel_d.to_csv(OUTPUT_DIR / "panel_d_single_gene_grade.csv", index=False)
panel_e.to_csv(OUTPUT_DIR / "panel_e_joint_models.csv", index=False)

display(panel_a)
display(panel_b)
display(panel_d)
display(panel_e)
print("TRACERx structural audit: PASS")


,gene,PC1_loading
0,BAK1,0.469494
1,BAX,0.569554
2,BID,0.603483
3,SLC7A11,0.301648


,score,grade_beta,stage_beta,punctuated_beta,grade_p,stage_p,punctuated_p,score_label
0,Equal4,0.172936,0.310398,0.467954,0.039400,0.033701,0.041285,Equal4
1,PCA1_4,0.173155,0.300864,0.420162,0.025109,0.048305,0.057593,PCA1
2,BAK1_centered,0.201766,0.335847,0.420317,0.004964,0.023892,0.054859,BAK1-centered
3,Apoptotic3,0.180987,0.291222,0.338755,0.016956,0.072678,0.117908,Core3
4,BAK1_SLC7A11,0.154538,0.292817,0.504109,0.340203,0.187890,0.064062,BAK1 + SLC7A11
5,BAK1_only,0.183187,0.274933,0.267161,0.013160,0.051407,0.290132,BAK1 only


,component,n,beta,se,p_value,ci_low,ci_high
0,BAK1,73,0.183187,0.073883,0.013160,0.038378,0.327996
1,BAX,73,0.114199,0.076150,0.133705,-0.035053,0.263451
2,BID,73,0.099009,0.078332,0.206245,-0.054520,0.252537
3,SLC7A11,73,0.042021,0.434526,0.922961,-0.809635,0.893676
4,Equal4,73,0.172936,0.083950,0.039400,0.008396,0.337475


,endpoint,component,n,beta,se,p_value,ci_low,ci_high
0,Grade,BAK1,73,0.161947,0.081826,0.047798,0.001571,0.322323
1,Grade,Rest3,73,0.109503,0.118601,0.355853,-0.122949,0.341956
2,Punctuated evolution,BAK1,73,0.294788,0.362124,0.415615,-0.414963,1.004538
3,Punctuated evolution,Rest3,73,0.735628,0.439783,0.094385,-0.126331,1.597587


TRACERx structural audit: PASS


## Stream the public JAVELIN molecular table


In [5]:
from pathlib import Path
import hashlib
import os
import re
import warnings

import numpy as np
import pandas as pd
import statsmodels.api as sm
from openpyxl import load_workbook
from pyxlsb import open_workbook as open_xlsb
from statsmodels.duration.hazard_regression import PHReg
from scipy.stats import norm


PROJECT_ROOT = Path(
    os.environ.get("MITOXY_PROJECT_ROOT", "/scratch/ar58064/mitoxy")
).resolve()
MANIFEST_PATH = (
    PROJECT_ROOT / "data" / "provenance" /
    "paper_public_source_manifest.csv"
)
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "figure_6_clean"
CACHE_DIR = PROJECT_ROOT / "data" / "processed" / "figure_6_public_cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(MANIFEST_PATH)


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def source_path(dataset, role):
    match = manifest.loc[
        manifest["dataset"].eq(dataset)
        & manifest["role"].eq(role)
    ]
    if len(match) != 1:
        raise RuntimeError(
            f"Expected one source for ({dataset!r}, {role!r}); "
            f"found {len(match)}"
        )
    row = match.iloc[0]
    path = PROJECT_ROOT / str(row["relative_path"])
    if not path.is_file():
        raise FileNotFoundError(path)
    if path.stat().st_size != int(row["size_bytes"]):
        raise RuntimeError(f"Source-size mismatch: {path}")
    if sha256_file(path) != str(row["sha256"]):
        raise RuntimeError(f"Source SHA-256 mismatch: {path}")
    return path


RAW = {
    "javelin": source_path(
        "JAVELIN Renal 101", "clinical and molecular tables"
    ),
    "checkmate": source_path(
        "CheckMate 025", "clinical and molecular tables"
    ),
}

display(pd.DataFrame({"analysis_key": RAW.keys(), "path": RAW.values()}))
print("Public-source integrity audit: PASS")


,analysis_key,path
0,javelin,/scratch/ar58064/mitoxy/data/raw/javelin_renal...
1,checkmate,/scratch/ar58064/mitoxy/data/raw/checkmate_025...


Public-source integrity audit: PASS


In [6]:
MITOXY_GENES = ["BAK1", "BAX", "BID", "SLC7A11"]
CORE3_GENES = ["BAK1", "BAX", "BID"]
MYELOID_GENES = ["MYD88", "TNF", "TLR2", "TLR4"]
SPATIAL4_GENES = ["MAPKAP1", "MLST8", "MTOR", "RICTOR"]
ANGIO_GENES = ["VEGFA", "KDR", "ESM1", "PECAM1", "ANGPTL4", "CD34"]
JAVELIN26_GENES = [
    "CD3G", "CD3E", "CD8B", "THEMIS", "TRAT1", "GRAP2", "CD247",
    "CD2", "CD96", "PRF1", "CD6", "IL7R", "ITK", "GPR18", "EOMES",
    "SIT1", "NLRC3", "CD244", "KLRD1", "SH2D1A", "CCL5", "XCL2",
    "CST7", "GFI1", "KCNA3", "PSTPIP1",
]

JAVELIN_TARGET_GENES = list(dict.fromkeys(
    MITOXY_GENES + MYELOID_GENES + SPATIAL4_GENES +
    ANGIO_GENES + JAVELIN26_GENES
))
CHECKMATE_TARGET_GENES = list(dict.fromkeys(
    MITOXY_GENES + MYELOID_GENES + SPATIAL4_GENES
))


def normalized(text):
    return re.sub(r"[^a-z0-9]+", " ", str(text).lower()).strip()


def canonical_id(value):
    if pd.isna(value):
        return ""
    text = re.sub(r"[^A-Za-z0-9]", "", str(value)).upper()
    if len(text) == 33 and text.startswith("X"):
        text = text[1:]
    return text


def unique_headers(values):
    output, counts = [], {}
    for index, value in enumerate(values):
        base = str(value).strip() if value not in (None, "") else f"column_{index}"
        counts[base] = counts.get(base, 0) + 1
        output.append(base if counts[base] == 1 else f"{base}_{counts[base]}")
    return output


def zscore_series(values):
    values = pd.to_numeric(values, errors="coerce").astype(float)
    standard_deviation = values.std(ddof=0)
    if not np.isfinite(standard_deviation) or standard_deviation == 0:
        raise ValueError("Cannot standardize a constant or invalid variable")
    return (values - values.mean()) / standard_deviation


def zscore_frame(frame):
    result = frame.apply(pd.to_numeric, errors="coerce").astype(float)
    standard_deviation = result.std(axis=0, ddof=0).replace(0, np.nan)
    result = result.subtract(result.mean(axis=0), axis=1).divide(
        standard_deviation, axis=1
    )
    if result.isna().any().any():
        bad = result.columns[result.isna().any()].tolist()
        raise ValueError(f"Cannot standardize genes with missing/invalid values: {bad}")
    return result


def bh_adjust(values):
    values = np.asarray(values, dtype=float)
    adjusted = np.full(values.shape, np.nan)
    valid = np.flatnonzero(np.isfinite(values))
    if len(valid) == 0:
        return adjusted
    order = valid[np.argsort(values[valid])]
    ranked = values[order] * len(order) / np.arange(1, len(order) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    adjusted[order] = np.minimum(ranked, 1.0)
    return adjusted


def resolve_numbered_sheet(sheet_names, number):
    patterns = [
        rf"(?:table|supplementary table|supp table|st|s)\s*0*{number}(?:\D|$)",
        rf"^0*{number}(?:\D|$)",
    ]
    matches = []
    for sheet in sheet_names:
        name = normalized(sheet)
        if any(re.search(pattern, name) for pattern in patterns):
            matches.append(sheet)
    if len(matches) == 1:
        return matches[0]
    contains = [sheet for sheet in sheet_names if str(number) in str(sheet)]
    if len(contains) == 1:
        return contains[0]
    raise RuntimeError(
        f"Could not uniquely resolve Supplementary Table {number}. "
        f"Candidates={matches or contains}; sheets={sheet_names}"
    )


def select_column(columns, alternatives, exclude=(), required=True):
    normalized_columns = {column: normalized(column) for column in columns}
    exclude = [normalized(term) for term in exclude]
    scored = []
    for column, text in normalized_columns.items():
        if any(term in text for term in exclude):
            continue
        best = 0
        for alternative in alternatives:
            terms = [normalized(term) for term in alternative]
            if all(term in text for term in terms):
                best = max(best, 100 + 10 * len(terms) - len(text))
        if best:
            scored.append((best, column))
    if not scored:
        if required:
            raise KeyError(
                f"No column matched {alternatives}. Available: {list(columns)}"
            )
        return None
    return max(scored, key=lambda item: (item[0], str(item[1])))[1]


def parse_event(values, column_name):
    numeric = pd.to_numeric(values, errors="coerce")
    observed = set(numeric.dropna().unique())
    name = normalized(column_name)
    if observed and observed.issubset({0, 1}):
        return 1 - numeric if ("censor" in name or "cnsr" in name) else numeric
    text = values.astype(str).str.strip().str.casefold()
    result = pd.Series(np.nan, index=values.index, dtype=float)
    result.loc[text.str.contains(
        r"death|dead|progress|event|yes|^1$", regex=True, na=False
    )] = 1.0
    result.loc[text.str.contains(
        r"censor|alive|no event|no progression|^0$", regex=True, na=False
    )] = 0.0
    return result


def parse_response(values):
    numeric = pd.to_numeric(values, errors="coerce")
    observed = set(numeric.dropna().unique())
    if observed and observed.issubset({0, 1}):
        return numeric.astype(float)
    text = values.astype(str).str.strip().str.upper()
    result = pd.Series(np.nan, index=values.index, dtype=float)
    result.loc[text.str.contains(
        r"COMPLETE RESPONSE|PARTIAL RESPONSE|^CR$|^PR$|RESPONDER",
        regex=True, na=False,
    )] = 1.0
    result.loc[text.str.contains(
        r"STABLE|PROGRESSIVE|^SD$|^PD$|NON.?RESPONDER",
        regex=True, na=False,
    )] = 0.0
    return result


def prepare_design(data, score_column, categorical_covariates=(),
                   numeric_covariates=()):
    required = [
        score_column, "treatment", *categorical_covariates,
        *numeric_covariates,
    ]
    complete = data[required].replace([np.inf, -np.inf], np.nan).dropna().copy()
    design = pd.DataFrame(index=complete.index)
    design["score"] = pd.to_numeric(complete[score_column], errors="coerce")
    design["treatment"] = pd.to_numeric(complete["treatment"], errors="coerce")
    for column in numeric_covariates:
        design[column] = zscore_series(complete[column])
    if categorical_covariates:
        dummies = pd.get_dummies(
            complete[list(categorical_covariates)].astype(str),
            prefix=list(categorical_covariates), drop_first=True, dtype=float,
        )
        design = pd.concat([design, dummies], axis=1)
    design["score_by_treatment"] = design["score"] * design["treatment"]
    return complete, design.astype(float)


def cox_interaction(data, score_column, time_column, event_column,
                    categorical_covariates=(), numeric_covariates=()):
    required = [time_column, event_column]
    analysis = data.dropna(subset=required).copy()
    complete, design = prepare_design(
        analysis, score_column, categorical_covariates, numeric_covariates
    )
    outcomes = analysis.loc[complete.index, [time_column, event_column]].astype(float)
    valid = outcomes[time_column].gt(0) & outcomes[event_column].isin([0, 1])
    outcomes, design = outcomes.loc[valid], design.loc[valid]
    fit = PHReg(
        outcomes[time_column], design,
        status=outcomes[event_column], ties="breslow",
    ).fit(disp=0)
    index = design.columns.get_loc("score_by_treatment")
    coefficient = float(fit.params[index])
    standard_error = float(fit.bse[index])
    return {
        "n": len(outcomes),
        "events": int(outcomes[event_column].sum()),
        "effect": float(np.exp(coefficient)),
        "ci_low": float(np.exp(coefficient - 1.96 * standard_error)),
        "ci_high": float(np.exp(coefficient + 1.96 * standard_error)),
        "p_value": float(fit.pvalues[index]),
    }


def cox_treatment_effect(data, time_column="pfs", event_column="pfs_event"):
    complete = data[[time_column, event_column, "treatment"]].dropna().copy()
    complete = complete.loc[
        complete[time_column].gt(0) & complete[event_column].isin([0, 1])
    ]
    design = complete[["treatment"]].astype(float)
    fit = PHReg(
        complete[time_column].astype(float), design,
        status=complete[event_column].astype(float), ties="breslow",
    ).fit(disp=0)
    coefficient, standard_error = float(fit.params[0]), float(fit.bse[0])
    return {
        "n": len(complete),
        "events": int(complete[event_column].sum()),
        "effect": float(np.exp(coefficient)),
        "ci_low": float(np.exp(coefficient - 1.96 * standard_error)),
        "ci_high": float(np.exp(coefficient + 1.96 * standard_error)),
        "p_value": float(fit.pvalues[0]),
    }


def logistic_interaction(data, score_column, response_column="orr"):
    analysis = data.dropna(subset=[response_column]).copy()
    complete, design = prepare_design(analysis, score_column)
    outcome = analysis.loc[complete.index, response_column].astype(float)
    valid = outcome.isin([0, 1])
    outcome, design = outcome.loc[valid], design.loc[valid]
    design = sm.add_constant(design, has_constant="add")
    fit = sm.Logit(outcome, design).fit(disp=False, maxiter=500)
    coefficient = float(fit.params["score_by_treatment"])
    standard_error = float(fit.bse["score_by_treatment"])
    return {
        "n": len(outcome),
        "events": np.nan,
        "effect": float(np.exp(coefficient)),
        "ci_low": float(np.exp(coefficient - 1.96 * standard_error)),
        "ci_high": float(np.exp(coefficient + 1.96 * standard_error)),
        "p_value": float(fit.pvalues["score_by_treatment"]),
    }


def validate_intervals(table, estimate):
    if not table[estimate].gt(0).all():
        raise ValueError(f"Non-positive estimates in {estimate}")
    if not table["ci_low"].gt(0).all():
        raise ValueError("Non-positive confidence limit")
    if not table["ci_high"].gt(table["ci_low"]).all():
        raise ValueError("Invalid confidence-interval ordering")
    if not (
        table["ci_low"].le(table[estimate])
        & table[estimate].le(table["ci_high"])
    ).all():
        raise ValueError("Estimate falls outside its confidence interval")


In [7]:
JAVELIN_CLINICAL_CACHE = CACHE_DIR / "javelin_table11_clinical.csv"
JAVELIN_EXPRESSION_CACHE = CACHE_DIR / "javelin_table13_target_expression.csv"


def xlsb_values(cell_row):
    return [cell.v for cell in cell_row]


def reconstruct_javelin_clinical(path):
    excel = pd.ExcelFile(path, engine="pyxlsb")
    sheet = resolve_numbered_sheet(excel.sheet_names, 11)
    rows = []
    with open_xlsb(path) as workbook:
        with workbook.get_sheet(sheet) as worksheet:
            for row in worksheet.rows():
                rows.append(xlsb_values(row))
    header_scores = []
    for index, row in enumerate(rows[:40]):
        tokens = {normalized(value) for value in row if value not in (None, "")}
        score = sum(any(key in token for token in tokens) for key in [
            "id", "pfs", "age", "sex", "trt", "pdl1", "cluster",
        ])
        header_scores.append((score, index))
    score, header_index = max(header_scores)
    if score < 4:
        raise RuntimeError(
            f"Could not identify the JAVELIN clinical header in {sheet!r}"
        )
    width = max(len(row) for row in rows[header_index:])
    header = unique_headers(rows[header_index] + [None] * (width - len(rows[header_index])))
    body = [row + [None] * (width - len(row)) for row in rows[header_index + 1:]]
    table = pd.DataFrame(body, columns=header).dropna(how="all").dropna(axis=1, how="all")
    id_column = select_column(table.columns, [["id"]])
    table["canonical_id"] = table[id_column].map(canonical_id)
    table = table.loc[table["canonical_id"].str.len().gt(0)].drop_duplicates("canonical_id")
    print("JAVELIN clinical sheet:", sheet)
    print("JAVELIN clinical rows:", len(table))
    return table


def reconstruct_javelin_expression(path, clinical_ids, targets):
    excel = pd.ExcelFile(path, engine="pyxlsb")
    sheet = resolve_numbered_sheet(excel.sheet_names, 13)
    targets = set(targets)
    first_rows, target_rows = [], {}
    row_count = 0
    with open_xlsb(path) as workbook:
        with workbook.get_sheet(sheet) as worksheet:
            for row_index, row in enumerate(worksheet.rows()):
                values = xlsb_values(row)
                row_count += 1
                if row_index < 30:
                    first_rows.append(values)
                gene = next((
                    str(value).strip().upper()
                    for value in values[:12]
                    if str(value).strip().upper() in targets
                ), None)
                if gene is not None:
                    target_rows.setdefault(gene, []).append(values)
    header_candidates = []
    for index, row in enumerate(first_rows):
        overlap = sum(canonical_id(value) in clinical_ids for value in row)
        header_candidates.append((overlap, index, row))
    overlap, header_index, header = max(header_candidates, key=lambda item: item[0])
    if overlap < 100:
        raise RuntimeError(
            f"JAVELIN Table 13 identifier overlap is too small: {overlap}"
        )
    positions = [
        (index, canonical_id(value))
        for index, value in enumerate(header)
        if canonical_id(value) in clinical_ids
    ]
    expression = pd.DataFrame(index=[sample for _, sample in positions])
    for gene, rows in target_rows.items():
        vectors = []
        for row in rows:
            vectors.append([
                pd.to_numeric(row[index], errors="coerce")
                if index < len(row) else np.nan
                for index, _ in positions
            ])
        expression[gene] = np.nanmean(np.asarray(vectors, dtype=float), axis=0)
    expression.index.name = "canonical_id"
    expression = expression.reset_index().drop_duplicates("canonical_id")
    print("JAVELIN expression sheet:", sheet)
    print("Rows streamed:", row_count)
    print("Detected identifier row:", header_index)
    print("Identifier overlap:", overlap)
    print("Target genes recovered:", sorted(target_rows))
    return expression


if JAVELIN_CLINICAL_CACHE.exists():
    javelin_clinical = pd.read_csv(JAVELIN_CLINICAL_CACHE, dtype=str)
    print("Loading cached JAVELIN clinical table.")
else:
    javelin_clinical = reconstruct_javelin_clinical(RAW["javelin"])
    javelin_clinical.to_csv(JAVELIN_CLINICAL_CACHE, index=False)

clinical_ids = set(javelin_clinical["canonical_id"].dropna().map(canonical_id))
if JAVELIN_EXPRESSION_CACHE.exists():
    javelin_expression = pd.read_csv(JAVELIN_EXPRESSION_CACHE)
    cached_genes = set(javelin_expression.columns) - {"canonical_id"}
    cached_overlap = len(set(javelin_expression["canonical_id"].map(canonical_id)) & clinical_ids)
    if not set(JAVELIN_TARGET_GENES).issubset(cached_genes) or cached_overlap < 100:
        print("Discarding incomplete JAVELIN expression cache.")
        JAVELIN_EXPRESSION_CACHE.unlink()
        javelin_expression = reconstruct_javelin_expression(
            RAW["javelin"], clinical_ids, JAVELIN_TARGET_GENES
        )
        javelin_expression.to_csv(JAVELIN_EXPRESSION_CACHE, index=False)
    else:
        print("Loading cached JAVELIN target-expression table.")
else:
    javelin_expression = reconstruct_javelin_expression(
        RAW["javelin"], clinical_ids, JAVELIN_TARGET_GENES
    )
    javelin_expression.to_csv(JAVELIN_EXPRESSION_CACHE, index=False)

javelin_expression["canonical_id"] = javelin_expression["canonical_id"].map(canonical_id)
javelin_clinical["canonical_id"] = javelin_clinical["canonical_id"].map(canonical_id)
missing = sorted(set(JAVELIN_TARGET_GENES) - set(javelin_expression.columns))
if missing:
    raise KeyError(f"JAVELIN Table 13 is missing target genes: {missing}")

print("JAVELIN clinical identifiers:", len(clinical_ids))
print("JAVELIN expression identifiers:", javelin_expression["canonical_id"].nunique())
print("JAVELIN clinical/expression overlap:", len(
    set(javelin_expression["canonical_id"]) & clinical_ids
))


Loading cached JAVELIN clinical table.
Loading cached JAVELIN target-expression table.
JAVELIN clinical identifiers: 886
JAVELIN expression identifiers: 726
JAVELIN clinical/expression overlap: 726


In [8]:
# The public JAVELIN clinical table uses compact field names in the released
# workbook. Resolve exact aliases first, then semantic fallbacks.
def exact_or_semantic(columns, exact_aliases, alternatives, exclude=()):
    lookup = {normalized(column): column for column in columns}
    for alias in exact_aliases:
        if normalized(alias) in lookup:
            return lookup[normalized(alias)]
    return select_column(columns, alternatives, exclude=exclude)


j_id = exact_or_semantic(javelin_clinical.columns, ["ID"], [["id"]])
j_pfs = exact_or_semantic(
    javelin_clinical.columns, ["PFS_P"],
    [["pfs", "time"], ["pfs", "month"], ["pfs"]],
    exclude=["cnsr", "censor", "event", "status"],
)
j_pfs_status = exact_or_semantic(
    javelin_clinical.columns, ["PFS_P_CNSR"],
    [["pfs", "cnsr"], ["pfs", "censor"], ["pfs", "event"], ["pfs", "status"]],
)
j_age = exact_or_semantic(javelin_clinical.columns, ["AGE"], [["age"]])
j_sex = exact_or_semantic(javelin_clinical.columns, ["SEX"], [["sex"], ["gender"]])
j_treatment = exact_or_semantic(
    javelin_clinical.columns, ["TRT01P"],
    [["treatment"], ["randomized", "arm"], ["arm"]],
)
j_pdl1 = exact_or_semantic(
    javelin_clinical.columns, ["PDL1FL"],
    [["pdl1"], ["pd", "l1"]],
)
j_cluster = exact_or_semantic(
    javelin_clinical.columns, ["TCGA_cluster"],
    [["tcga", "cluster"], ["molecular", "cluster"], ["cluster"]],
)

javelin_gene_values = javelin_expression.set_index("canonical_id")[JAVELIN_TARGET_GENES]
javelin_gene_values = javelin_gene_values.apply(pd.to_numeric, errors="coerce")
if javelin_gene_values.isna().any().any():
    raise ValueError("JAVELIN target-gene expression contains missing values")
javelin_gene_z = zscore_frame(javelin_gene_values)

javelin_scores = pd.DataFrame(index=javelin_gene_z.index)
javelin_scores["Mitoxy"] = zscore_series(javelin_gene_z[MITOXY_GENES].mean(axis=1))
javelin_scores["Core3"] = zscore_series(javelin_gene_z[CORE3_GENES].mean(axis=1))
javelin_scores["Myeloid trigger"] = zscore_series(
    javelin_gene_z[MYELOID_GENES].mean(axis=1)
)
javelin_scores["Spatial4"] = zscore_series(
    javelin_gene_z[SPATIAL4_GENES].mean(axis=1)
)
javelin_scores["ANGIO"] = zscore_series(javelin_gene_z[ANGIO_GENES].mean(axis=1))
javelin_scores["JAVELIN26"] = zscore_series(
    javelin_gene_z[JAVELIN26_GENES].mean(axis=1)
)
for gene in ["SLC7A11", "BID", "BAK1", "BAX", "RICTOR"]:
    javelin_scores[gene] = javelin_gene_z[gene]
javelin_scores.index.name = "canonical_id"

javelin = javelin_scores.reset_index().merge(
    javelin_clinical, on="canonical_id", how="inner", validate="one_to_one"
)
javelin["pfs"] = pd.to_numeric(javelin[j_pfs], errors="coerce")
javelin["pfs_event"] = parse_event(javelin[j_pfs_status], j_pfs_status)
javelin["age"] = pd.to_numeric(javelin[j_age], errors="coerce")
javelin["sex"] = javelin[j_sex].astype(str).str.strip()
javelin["pdl1"] = javelin[j_pdl1].astype(str).str.strip()
javelin["tcga_cluster"] = javelin[j_cluster].astype(str).str.strip()
javelin["treatment_label"] = javelin[j_treatment].astype(str).str.strip()
treatment_text = javelin["treatment_label"].str.casefold()
javelin["treatment"] = np.select(
    [
        treatment_text.str.contains(r"avelumab|axitinib|comb", regex=True),
        treatment_text.str.contains(r"sunitinib", regex=True),
    ],
    [1.0, 0.0], default=np.nan,
)

javelin_model = javelin.dropna(subset=[
    "pfs", "pfs_event", "age", "sex", "pdl1", "tcga_cluster", "treatment"
]).copy()
if not set(javelin_model["treatment"].unique()).issubset({0.0, 1.0}):
    raise ValueError("JAVELIN treatment coding failed")
if javelin_model["treatment"].nunique() != 2:
    raise ValueError("Both JAVELIN treatment arms were not recovered")

print("JAVELIN matched model rows:", len(javelin_model))
print("JAVELIN PFS events:", int(javelin_model["pfs_event"].sum()))
print("Treatment counts:")
display(javelin_model["treatment_label"].value_counts().to_frame("n"))
display(pd.DataFrame({
    "resolved_field": ["ID", "PFS", "PFS status", "age", "sex", "treatment", "PD-L1", "TCGA cluster"],
    "source_column": [j_id, j_pfs, j_pfs_status, j_age, j_sex, j_treatment, j_pdl1, j_cluster],
}))


JAVELIN matched model rows: 723
JAVELIN PFS events: 356
Treatment counts:


,n
treatment_label,
Sunitinib,371
Avelumab+Axitinib,352


,resolved_field,source_column
0,ID,ID
1,PFS,PFS_P
2,PFS status,PFS_P_CNSR
3,age,AGE
4,sex,SEX
5,treatment,TRT01P
6,PD-L1,PDL1FL
7,TCGA cluster,TCGA_cluster


In [9]:
OUTPUT_DIR=PROJECT_ROOT/"data"/"processed"/"supplementary_figure_s3"; OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

def parse_t(value):
    match=re.search(r"T([1-4])",str(value).upper()) if pd.notna(value) else None
    return float(match.group(1)) if match else np.nan

# S3A: patient-level Equal4 versus four-gene PC1 concordance.
complete=model_data[["Equal4","PCA1_4"]].dropna(); rho,p=spearmanr(complete.Equal4,complete.PCA1_4)
z=np.arctanh(np.clip(rho,-.999999,.999999)); se=1/np.sqrt(len(complete)-3)
panel_a=pd.DataFrame([{"comparison":"Equal4 vs PC1","n":len(complete),"rho":rho,"ci_low":np.tanh(z-1.96*se),
                       "ci_high":np.tanh(z+1.96*se),"p_value":p}])
panel_a.to_csv(OUTPUT_DIR/"panel_a_pc1_equal4.csv",index=False)


## S3B — E-MTAB Equal4/Core3/SLC7A11 clinical comparison


In [10]:
em_expr=source_path("E-MTAB-1980","ccRCC_exp_log_quantile_normalized.txt")
em_clin=source_path("E-MTAB-1980","Sato_2013_Supplementary_Table_1_patient_characteristics.xlsx")
raw=pd.read_csv(em_expr,sep="\t",index_col=0,low_memory=False)
probes={"BAK1":["A_23_P145357"],"BAX":["A_33_P3341144"],"BID":["A_24_P187948","A_33_P3227041"],"SLC7A11":["A_33_P3242623"]}
samples=[c for c in raw.columns if re.fullmatch(r"ccRCC-\d+",str(c),re.I)]; genes=pd.DataFrame(index=samples)
for gene,ids in probes.items(): genes[gene]=raw.loc[ids,samples].apply(pd.to_numeric,errors="coerce").mean(axis=0)
gz=genes.apply(zscore,axis=0); em=pd.DataFrame({"sample_id":samples,"Equal4":gz[GENES].mean(axis=1).values,
 "Core3":gz[CORE3].mean(axis=1).values,"SLC7A11":gz["SLC7A11"].values})
cl=pd.read_excel(em_clin,header=1,dtype=str); cl.columns=[str(x).strip() for x in cl.columns]
em=em.merge(pd.DataFrame({"sample_id":cl["sample ID"].astype(str).str.strip(),"grade":cl["Fuhrman grade"].map(parse_grade),
                          "t_category":cl["Stage at diagnosis"].map(parse_t)}),on="sample_id",how="left")
rows=[]
for score in ["Equal4","Core3","SLC7A11"]:
 for endpoint,label in [("grade","Grade"),("t_category","T category")]:
    c=em[[score,endpoint]].dropna(); r,p=spearmanr(c[score],c[endpoint]); z=np.arctanh(np.clip(r,-.999999,.999999)); se=1/np.sqrt(len(c)-3)
    rows.append({"score":score,"endpoint":label,"n":len(c),"rho":r,"ci_low":np.tanh(z-1.96*se),"ci_high":np.tanh(z+1.96*se),"p_value":p})
panel_b=pd.DataFrame(rows); panel_b.to_csv(OUTPUT_DIR/"panel_b_emtab_components.csv",index=False); display(panel_b)


/scratch/ar58064/sulindac_tam/envs/sulindac-tam/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


,score,endpoint,n,rho,ci_low,ci_high,p_value
0,Equal4,Grade,99,0.320157,0.131022,0.486804,0.001236
1,Equal4,T category,101,0.285368,0.095237,0.455411,0.003820
2,Core3,Grade,99,0.291264,0.099574,0.462108,0.003445
3,Core3,T category,101,0.291788,0.102171,0.460944,0.003070
4,SLC7A11,Grade,99,0.153630,-0.045155,0.340712,0.128961
5,SLC7A11,T category,101,0.144539,-0.052384,0.330641,0.149274


## S3C — JAVELIN and CPTAC component-to-myeloid correlations


In [11]:
rows=[]
for score in ["Mitoxy","Core3","SLC7A11"]:
    c=javelin[[score,"Myeloid trigger"]].dropna(); r,p=spearmanr(c[score],c["Myeloid trigger"]); z=np.arctanh(np.clip(r,-.999999,.999999)); se=1/np.sqrt(len(c)-3)
    rows.append({"cohort":"JAVELIN101","score":"Equal4" if score=="Mitoxy" else score,"n":len(c),"rho":r,
                 "ci_low":np.tanh(z-1.96*se),"ci_high":np.tanh(z+1.96*se),"p_value":p})
cptac_rna=read_linkedomics(source_path("CPTAC ccRCC","tumor RNA expression"))
myeloid_genes=["MYD88","TNF","TLR2","TLR4"]; cz=cptac_rna.loc[GENES+myeloid_genes].T.apply(zscore,axis=0)
cs=pd.DataFrame({"Equal4":cz[GENES].mean(axis=1),"Core3":cz[CORE3].mean(axis=1),"SLC7A11":cz.SLC7A11,
                 "Myeloid trigger":cz[myeloid_genes].mean(axis=1)})
for score in ["Equal4","Core3","SLC7A11"]:
    c=cs[[score,"Myeloid trigger"]].dropna(); r,p=spearmanr(c[score],c["Myeloid trigger"]); z=np.arctanh(np.clip(r,-.999999,.999999)); se=1/np.sqrt(len(c)-3)
    rows.append({"cohort":"CPTAC","score":score,"n":len(c),"rho":r,"ci_low":np.tanh(z-1.96*se),"ci_high":np.tanh(z+1.96*se),"p_value":p})
panel_c=pd.DataFrame(rows); panel_c.to_csv(OUTPUT_DIR/"panel_c_myeloid_components.csv",index=False); display(panel_c)


,cohort,score,n,rho,ci_low,ci_high,p_value
0,JAVELIN101,Equal4,726,0.448136,0.388025,0.504451,3.752553e-37
1,JAVELIN101,Core3,726,0.471239,0.412624,0.525969,2.078596e-41
2,JAVELIN101,SLC7A11,726,0.177135,0.105733,0.246719,1.566226e-06
3,CPTAC,Equal4,110,0.329622,0.151742,0.486820,4.374167e-04
4,CPTAC,Core3,110,0.312761,0.133324,0.472344,8.789439e-04
5,CPTAC,SLC7A11,110,0.203003,0.016380,0.375957,3.342125e-02


## S3D–E — TRACERx architectures and TCGA joint models


In [12]:
panel_d=panel_b.copy() if False else globals()["panel_b"]
# Recover the TRACERx table produced before the E-MTAB table overwrote panel_b.
architecture_rows=[]
for score in score_names:
    architecture_rows.append({"score":score,"rho_with_equal4":spearmanr(model_data[score],model_data["Equal4"])[0],
      "grade_effect":fit_hc3(model_data,"grade",[f"{score}_sd","purity","age"],f"{score}_sd")["beta"],
      "stage_effect":fit_hc3(model_data,"stage",[f"{score}_sd","purity","age"],f"{score}_sd")["beta"]})
panel_d=pd.DataFrame(architecture_rows); panel_d.to_csv(OUTPUT_DIR/"panel_d_tracerx_architectures.csv",index=False)

tcga_x=source_path("TCGA-KIRC","tumor RNA expression"); tcga_c=source_path("TCGA-KIRC","clinical API response")
x=pd.read_csv(tcga_x,sep="\t",index_col=0); x.index=x.index.astype(str).str.upper(); g=x.loc[GENES].T.apply(pd.to_numeric,errors="coerce")
g.index=g.index.astype(str).str.upper(); g=g.loc[g.index.str.match(r"^TCGA-.+-.+-01")]; g["sample_id"]=g.index.str[:12]; g=g.groupby("sample_id")[GENES].mean()
gz=g.apply(zscore,axis=0).reset_index(); obj=json.loads(tcga_c.read_text()); cases=obj.get("data",{}).get("hits",[]); clinical=[]
for case in cases:
    diag=(case.get("diagnoses") or [{}])[0]; demo=case.get("demographic") or {}
    clinical.append({"sample_id":case.get("submitter_id"),"grade":parse_grade(diag.get("tumor_grade")),"stage":parse_stage(diag.get("ajcc_pathologic_stage")),
                     "age":pd.to_numeric(diag.get("age_at_diagnosis"),errors="coerce")/365.25,"male":float(str(demo.get("gender","")).lower()=="male")})
tc=gz.merge(pd.DataFrame(clinical),on="sample_id",how="left")
def joint_rows(outcome):
    complete=tc[[outcome,*GENES,"age","male"]].dropna(); X=sm.add_constant(complete[[*GENES,"age","male"]],has_constant="add")
    fit=sm.OLS(complete[outcome],X).fit(cov_type="HC3",use_t=False); out=[]
    for gene in GENES:
        ci=fit.conf_int().loc[gene]; out.append({"endpoint":outcome.title(),"gene":gene,"n":len(complete),"effect":fit.params[gene],"se":fit.bse[gene],
                                                "ci_low":ci.iloc[0],"ci_high":ci.iloc[1],"p_value":fit.pvalues[gene]})
    return out
panel_e=pd.DataFrame(joint_rows("grade")+joint_rows("stage")); panel_e.to_csv(OUTPUT_DIR/"panel_e_tcga_joint.csv",index=False); display(panel_d); display(panel_e)


/tmp/ipykernel_1434645/81045118.py:21: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  fit=sm.OLS(complete[outcome],X).fit(cov_type="HC3",use_t=False); out=[]
/tmp/ipykernel_1434645/81045118.py:21: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  fit=sm.OLS(complete[outcome],X).fit(cov_type="HC3",use_t=False); out=[]


,score,rho_with_equal4,grade_effect,stage_effect
0,Equal4,1.000000,0.172936,0.310398
1,PCA1_4,0.994848,0.173155,0.300864
2,BAK1_centered,0.927126,0.201766,0.335847
3,Apoptotic3,0.981797,0.180987,0.291222
4,BAK1_SLC7A11,0.744878,0.154538,0.292817
5,BAK1_only,0.632975,0.183187,0.274933


,endpoint,gene,n,effect,se,ci_low,ci_high,p_value
0,Grade,BAK1,428,0.210990,0.037843,0.136819,0.285160,2.468924e-08
1,Grade,BAX,428,0.029386,0.036004,-0.041180,0.099953,4.143836e-01
2,Grade,BID,428,0.101267,0.040201,0.022475,0.180060,1.176808e-02
3,Grade,SLC7A11,428,0.092719,0.035648,0.022851,0.162587,9.295285e-03
4,Stage,BAK1,434,0.263907,0.057846,0.150530,0.377284,5.061884e-06
5,Stage,BAX,434,-0.149756,0.052884,-0.253406,-0.046106,4.628557e-03
6,Stage,BID,434,0.244910,0.056324,0.134517,0.355303,1.372434e-05
7,Stage,SLC7A11,434,-0.004806,0.055700,-0.113976,0.104363,9.312357e-01


In [13]:
contracts = {'panel_a_pc1_equal4.csv': ['comparison', 'n', 'rho', 'ci_low', 'ci_high', 'p_value'], 'panel_b_emtab_components.csv': ['score', 'endpoint', 'n', 'rho', 'ci_low', 'ci_high', 'p_value'], 'panel_c_myeloid_components.csv': ['cohort', 'score', 'n', 'rho', 'ci_low', 'ci_high', 'p_value'], 'panel_d_tracerx_architectures.csv': ['score', 'rho_with_equal4', 'grade_effect', 'stage_effect'], 'panel_e_tcga_joint.csv': ['endpoint', 'gene', 'n', 'effect', 'ci_low', 'ci_high', 'p_value']}
rows=[]
for filename,required in contracts.items():
    path=OUTPUT_DIR/filename
    table=pd.read_csv(path)
    missing=sorted(set(required)-set(table.columns))
    if missing: raise KeyError(f"{filename} missing {missing}")
    rows.append({"file":filename,"rows":len(table),"size_bytes":path.stat().st_size,"sha256":sha256_file(path)})
processed_manifest=pd.DataFrame(rows)
processed_manifest.to_csv(OUTPUT_DIR/"processed_manifest.csv",index=False)
display(processed_manifest)
print("Processed-file handoff: PASS")


,file,rows,size_bytes,sha256
0,panel_a_pc1_equal4.csv,1,136,42cd178d21395a868c994ade07f746b2d16da3fbab77e2...
1,panel_b_emtab_components.csv,6,636,1bb68062fac3ec54fb1fead9c0b0331a30bccd0d0bac28...
2,panel_c_myeloid_components.csv,6,643,e1b7a7f64c7e356c7c4d1e2cc30e0dade5ab9983a3e08a...
3,panel_d_tracerx_architectures.csv,6,441,557db81a5378660adc3ca227e7ccb61dba41a4bb9c48df...
4,panel_e_tcga_joint.csv,8,989,3aabd3cb2b84cc13d13e4db35ec71cd7023f535b128cbf...


Processed-file handoff: PASS
